# Data Analysis and Visualisation

## Lab 3: Clean new data

In [ ]:
import pandas as pd

### Part 1

#### Task 1: Load and look

In [ ]:
loans_raw = pd.read_csv('data/loans.csv')
loans = loans_raw.copy()

print(loans.shape)
loans.info()

(623, 9)
<class 'pandas.DataFrame'>
RangeIndex: 623 entries, 0 to 622
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   loan_id      623 non-null    str    
 1   member_id    623 non-null    str    
 2   title        623 non-null    str    
 3   genre        623 non-null    str    
 4   loan_date    623 non-null    str    
 5   return_date  593 non-null    str    
 6   late_fee     623 non-null    str    
 7   returned     623 non-null    str    
 8   notes        0 non-null      float64
dtypes: float64(1), str(8)
memory usage: 43.9 KB


Problems I can see: 

- columns `loan_date`, `return_date` are text, but should be dates. 
- Column `late_fee` is also text, but should be a number.
- Column `returned` is text too, but should be `True/False`.
- Column `notes` is completely empty (0 non-null). 
- Column `return_date` has 30 missing values.

#### Task 2: Clean the genre

In [ ]:
print(loans['genre'].nunique())

loans['genre'] = loans['genre'].str.strip().str.lower()
print(loans['genre'].nunique())
print(list(loans['genre'].unique()))

28
9
['romance', 'crime', 'fantasy', 'children', "children's", 'history', 'sci-fi', 'science fiction', 'sci fi']


After `strip` and `lower` there are 9 values left for 6 genres: "children's", "science fiction" and "sci fi" are different spellings of genres that exist already.

In [ ]:
loans['genre'] = loans['genre'].replace({"children's": 'children',
                                         'science fiction': 'sci-fi',
                                         'sci fi': 'sci-fi'})
loans['genre'] = loans['genre'].str.title()
print(loans['genre'].value_counts())

genre
Crime       163
Children    135
Fantasy      90
Sci-Fi       90
Romance      86
History      59
Name: count, dtype: int64


Now we have 6 unique genres, and `.value_counts()` shows the number of loans in each genre.

#### Task 3: Yes and no

In [ ]:
print(loans['returned'].nunique())
print(loans['returned'].str.strip().str.lower().unique())

10
<StringArray>
['yes', 'y', 'no', 'n']
Length: 4, dtype: str


10 variants of "yes"/"no" (Yes, YES, y, N ...). After strip and lower there are four: "yes", "y", "no", "n". `map` turns them into `True/False`, and anything I forgot would become `NaN`.

In [ ]:
loans['returned'] = loans['returned'].str.strip().str.lower().map(
    {'yes': True, 'y': True, 'no': False, 'n': False})
print("Missed values:", loans['returned'].isna().sum())
display(loans['returned'].value_counts())

Missed values: 0


returned
True     593
False     30
Name: count, dtype: int64

#### Task 4: The late fee is text

In [ ]:
# This will fail
loans['late_fee'].astype(float)
# `astype(float)` fails because python cannot read the "kr", the space and the decimal comma as part of a number.

ValueError: could not convert string to float: '7,50 kr'

In [ ]:
loans['late_fee'] = (loans['late_fee']
                     .str.replace('kr', '')
                     .str.replace(' ', '')
                     .str.replace(',', '.')
                     .astype(float))
display(loans['late_fee'].describe())
print("The total of all fees:", loans['late_fee'].sum())

count    623.000000
mean      12.323435
std       16.106338
min        0.000000
25%        0.000000
50%        0.000000
75%       25.000000
max       47.500000
Name: late_fee, dtype: float64

The total of all fees: 7677.5


#### Task 5: Dates

In [ ]:
loans['loan_date'] = pd.to_datetime(loans['loan_date'], errors='coerce')

print("The number of dates that became NaT:", loans['loan_date'].isna().sum())
print(f"First loan date: {loans['loan_date'].min()}")
print(f"Last loan date: {loans['loan_date'].max()}")

The number of dates that became NaT: 3
First loan date: 2026-01-05 00:00:00
Last loan date: 2026-06-27 00:00:00


In [ ]:
per_month = loans['loan_date'].dt.to_period('M').value_counts().sort_index()
print(per_month)
print(per_month.sum(), len(loans))

loan_date
2026-01     96
2026-02    100
2026-03    111
2026-04    106
2026-05    108
2026-06     99
Freq: M, Name: count, dtype: int64
620 623


3 dates could not be read and became `NaT`. The monthly counts add up to 620, which is 623 minus those 3, so the `NaT` rows are the ones missing from the months.

#### Task 6: Duplicates

In [ ]:
print("Duplicates in the original dataset:", loans_raw.duplicated().sum())
print("Duplicates in the cleaned dataset:", loans.duplicated().sum())
print("Number of rows with the same loan_id:", loans.duplicated(subset='loan_id').sum())

Duplicates in the original dataset: 12
Duplicates in the cleaned dataset: 18
Number of rows with the same loan_id: 18


The raw table has 12 exact duplicates. After the cleaning there are 18, because 6 pairs only differed in spelling (e.g. "ROMANCE" vs "romance"). They became identical once the text was cleaned. This is why duplicates are removed after cleaning.

In [ ]:
fee_before = loans['late_fee'].sum()
loans = loans.drop_duplicates().reset_index(drop=True)
fee_after = loans['late_fee'].sum()

fee_diff = fee_before - fee_after
fee_diff_rel = fee_diff / fee_before * 100

In [ ]:
print("The shape of the DF after removing duplicates:", loans.shape)
print(f"The duplicate rows added {fee_diff:.2f} kr to the total of the late fees, which is ca {fee_diff_rel:.2f}%.")

The shape of the DF after removing duplicates: (605, 9)
The duplicate rows added 230.00 kr to the total of the late fees, which is ca 3.00%.
